This notebook loads the donor-level single-nucleus transcriptomic count matrices from Dumitru et al., reads each donor’s matrix.mtx, barcodes.tsv, and features.tsv files, and constructs a single AnnData object integrated across all donors. It adds donor-level metadata, concatenates all samples into one dataset, and writes the combined object to an .h5ad file for downstream preprocessing and analysis.

Data source: Dumitru et al. raw snRNA-seq data are available [here](https://zenodo.org/records/14879680) and at DOI 10.5281/zenodo.14879680.

Input: donor-level raw count matrices in ../data/dumitru/single_nucleus/raw/  
Output: combined .h5ad file written to ../data/dumitru/single_nucleus/h5ad/

In [ ]:
import pandas as pd
import anndata as ad
from pathlib import Path
from scipy.io import mmread

In [ ]:
def create_combined_anndata(directory: str):
    data_path = Path(directory)
    adatas = []

    matrix_files = data_path.glob('*_matrix.mtx')
    prefixes = sorted([f.name.removesuffix('_matrix.mtx') for f in matrix_files])

    for prefix in prefixes:
        matrix_file = data_path / f'{prefix}_matrix.mtx'
        features_file = data_path / f'{prefix}_features.tsv'
        barcodes_file = data_path / f'{prefix}_barcodes.tsv'

        if not (features_file.exists() and barcodes_file.exists()):
            continue
        
        counts = mmread(matrix_file).tocsr().T
        features = pd.read_csv(features_file, sep='\t', header=None, names=['gene_symbols'])
        barcodes = pd.read_csv(barcodes_file, sep='\t', header=None, names=['barcode'])

        adata = ad.AnnData(X=counts)
        adata.var_names = features['gene_symbols'].values
        adata.obs_names = barcodes['barcode'].values
        adata.obs['patient_id'] = prefix

        print(f"Loaded sample: {prefix}")

        adatas.append(adata)

    if not adatas:
        return None

    return ad.concat(adatas, join='outer', index_unique='-')

In [ ]:
adata = create_combined_anndata('../data/dumitru/single_nucleus/raw/')
adata

In [ ]:
# Extract metadata
meta = pd.read_excel('../data/dumitru/table_S1.xlsx', header=1, nrows=26)
meta.columns = meta.columns.str.strip()
meta = meta[['Donor', 'Age', 'SEX', 'Cause/manner of death', 'Neurological or other relevant diagnosis', 'Sample Prep Protocol']]
meta.columns = ['donor', 'age', 'sex', 'death', 'diagnosis', 'sample_prep']
meta['donor'] = meta['donor'].astype(str)
meta = meta.set_index('donor')
meta

In [ ]:
# Extract donor number from patient_id
adata.obs['donor'] = (
    adata.obs['patient_id']
      .astype(str)
      .str.slice(2)            # drop first 2 letters
      .str.split('_').str[0]   # take substring before underscore
)

# Join metadata to adata.obs
adata.obs = adata.obs.join(meta, on='donor')
adata.obs

In [ ]:
adata

In [ ]:
adata.write_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_raw_concat.h5ad')